In [13]:
import re
import json
import os
import pandas as pd

MAX_WORD_CHUNK = 450


def read_json_file(text_json_path : str):
    """
    Read Json file of one video
    """
    json_object = pd.read_json(text_json_path, lines=True)
    return json_object

def word_count(text : str):
    return len(re.findall(r'\w+', text))

In [14]:
json_object = read_json_file(
    text_json_path = r"C:\Users\axelo\Documents\Back up documents\Documents\PGE5\GEN AI\project\RepWise\data\corpus\passages\_-XnY-bwD4A.jsonl"
)

In [15]:
json_object

,video_id,passage_id,start,end,url_t,text,flags
0,_-XnY-bwD4A,_-XnY-bwD4A_p0001,0,14,https://youtu.be/_-XnY-bwD4A?t=0,"De tous les muscles du corps, il y en a un qui...",[]
1,_-XnY-bwD4A,_-XnY-bwD4A_p0002,14,29,https://youtu.be/_-XnY-bwD4A?t=14,c'est devenu l'un de mes points forts. Le prob...,[]
2,_-XnY-bwD4A,_-XnY-bwD4A_p0003,29,36,https://youtu.be/_-XnY-bwD4A?t=29,je vais vous partager sep conseils plus un bon...,[]
3,_-XnY-bwD4A,_-XnY-bwD4A_p0004,36,49,https://youtu.be/_-XnY-bwD4A?t=36,"Et honnêtement, si vous appliquez correctement...",[]
4,_-XnY-bwD4A,_-XnY-bwD4A_p0005,49,55,https://youtu.be/_-XnY-bwD4A?t=49,"Quand on parle de muscler son dos, on utilise ...",[]
...,...,...,...,...,...,...,...
69,_-XnY-bwD4A,_-XnY-bwD4A_p0070,904,922,https://youtu.be/_-XnY-bwD4A?t=904,"Ligne de force, bras de levier, unilatéral, sé...",[]
70,_-XnY-bwD4A,_-XnY-bwD4A_p0071,922,938,https://youtu.be/_-XnY-bwD4A?t=922,progressive pour progresser sur n'importe quel...,[]
71,_-XnY-bwD4A,_-XnY-bwD4A_p0072,938,950,https://youtu.be/_-XnY-bwD4A?t=938,bras de levier. Je savais pas ce que c'était u...,[]
72,_-XnY-bwD4A,_-XnY-bwD4A_p0073,950,966,https://youtu.be/_-XnY-bwD4A?t=950,C'est évidemment quelque chose que je ne fais ...,[]


In [34]:
type(re.search(r"\d{2}$", json_object.iloc[0]["passage_id"]).group())

str

In [20]:
json_object.iloc[0]["text"]

"De tous les muscles du corps, il y en a un qui permet à lui seul de paraître instantanément plus musclé. Et c'est pas les abdos, c'est pas les biceps, c'est même pas les pectoraux, c'est bien sûr le dos. Et personnellement, j'ai mis des années à comprendre comment bien le travailler, mais depuis que j'ai fait les ajustements nécessaires, mon dos,"

In [ ]:
n_rows = len(json_object)
start_index = 0
chunk_id = 1
chunk_dict = {}

MAX_WORD_CHUNK = 450
OVERLAP_PASSAGES = 1

while start_index < n_rows:
    row_index = start_index
    word_total = 0
    texts, flags, passage_ids = [], set(), []

    start_time = json_object.iloc[row_index]["start"]
    start_passage_id = re.search(r"_p(\d+)$", json_object.iloc[row_index]["passage_id"]).group()

    while row_index < n_rows:
        row = json_object.iloc[row_index]
        n_words = word_count(row["text"])

        if texts and word_total + n_words > MAX_WORD_CHUNK:
            break

        texts.append(row["text"])
        flags.update(row["flags"])
        passage_ids.append(row["passage_id"])
        word_total += n_words
        row_index += 1

    end_time = json_object.iloc[row_index - 1]["end"]
    end_time_passage_id = re.search(r"_p(\d+)$", json_object.iloc[row_index - 1]["passage_id"]).group()
    url = json_object.iloc[row_index - 1]["url_t"]
    video_url = re.match(r"[^?]+", url).group()
    video_id = json_object.iloc[row_index - 1]["video_id"]
    chunk_ID = video_id + f"_{str(start_passage_id)}" + f"_{str(end_time_passage_id)}"
    

    
    chunk_dict[f"chunk_{chunk_ID}"] = {
        "text": " ".join(texts),
        "flags": sorted(flags),
        "word_count": word_total,
        "passage_ids": passage_ids,
        "start": start_time,
        "end": end_time,
        "video_url": video_url,
        "video_id": video_id
    }
    chunk_id += 1

    if row_index >= n_rows:  
        break

    start_index = max(row_index - OVERLAP_PASSAGES, start_index + 1)



In [49]:
chunk_dict

{'chunk__-XnY-bwD4A__p0001__p0008': {'text': "De tous les muscles du corps, il y en a un qui permet à lui seul de paraître instantanément plus musclé. Et c'est pas les abdos, c'est pas les biceps, c'est même pas les pectoraux, c'est bien sûr le dos. Et personnellement, j'ai mis des années à comprendre comment bien le travailler, mais depuis que j'ai fait les ajustements nécessaires, mon dos, c'est devenu l'un de mes points forts. Le problème, c'est que la majorité des pratiquants entraînent leur dos depuis des années sans vraiment avoir de résultats significatifs. Parfois, c'est une mauvaise exécution, un mauvais angle de travail, un mauvais choix d'exercice, mais dans tous les cas, c'est la même chose. Il s'entraîne beaucoup mais le stimule très peu. Et dans cette vidéo, je vais vous partager sep conseils plus un bonus basé sur la biomécanique et ma propre expérience pour que vous puissiez réellement optimiser le travail du dos. Et honnêtement, si vous appliquez correctement tous ces 

In [54]:
def read_manifest(manifest_path : str, coverage_path : str):
    manifest_dataframe, coverage_dataframe = pd.read_csv(manifest_path), pd.read_csv(coverage_path)
    return manifest_dataframe, coverage_dataframe

In [55]:
manifest_df, coverage_df = read_manifest(r"C:\Users\axelo\Documents\Back up documents\Documents\PGE5\GEN AI\project\RepWise\data\corpus\manifest.csv",
r"C:\Users\axelo\Documents\Back up documents\Documents\PGE5\GEN AI\project\RepWise\data\corpus\coverage_matrix.csv")

In [53]:
manifest_df

,video_id,url,title,source_file,raw_sha256,upload_date,duration_s,caption_layout,punctuated,n_passages,n_words,n_corrections,sponsor_passages,scientific_references,n_references,reference_review_status,corpus_version
0,XFsbtGB1TXQ,https://youtu.be/XFsbtGB1TXQ,Ce format d'entraînement est parfait (Programm...,Ce format d'entraînement est parfait (Programm...,209e543113df26e2a0ed7931fc0047f64904fff518a8aa...,NaN,1131,rolling,True,89,4506,36,01:20|02:12|18:02,NaN,0,NaN,v1-2026-10-10
1,1OsDiVvQT7k,https://youtu.be/1OsDiVvQT7k,On s'est tous trompés sur le volume optimal po...,On s’est tous trompés sur le volume optimal po...,7716d9423067f6b6331ca85d50e441ae2db17f0fed2f4d...,NaN,832,rolling,True,66,3248,12,13:28,https://pmc.ncbi.nlm.nih.gov/articles/PMC88848...,3,mentioned,v1-2026-10-10
2,05WPpiDdC78,https://youtu.be/05WPpiDdC78,À quel niveau de musculation êtes-vous vraimen...,À quel niveau de musculation êtes-vous vraimen...,764f9a5cc038765ba6eba242b3f386d65ed01b5269f888...,NaN,1054,rolling,True,75,3969,2,06:07|06:24|17:13,NaN,0,NaN,v1-2026-10-10
3,_-XnY-bwD4A,https://youtu.be/_-XnY-bwD4A,7 conseils qui m'ont aidé à avoir un GROS DOS,7 conseils qui m’ont aidé à avoir un GROS DOS.txt,673e3f1be035aa53b2d35cba5a199e309ce236217b7cd3...,NaN,978,rolling,True,74,3927,42,11:21|11:35,NaN,0,NaN,v1-2026-10-10
4,cvfTTuOZKV8,https://youtu.be/cvfTTuOZKV8,"Comment j'ai atteint 9,5% de masse grasse en 3...","Comment j’ai atteint 9,5% de masse grasse en 3...",b7e04181f134013506a632e9538d3df907afc93d53a75d...,NaN,1026,rolling,True,88,4389,4,03:55|04:07|04:21,NaN,0,NaN,v1-2026-10-10
5,ZP0LLX_Srs8,https://youtu.be/ZP0LLX_Srs8,Est-ce que je regrette d'avoir perdu autant de...,Est-ce que je regrette d'avoir perdu autant de...,fea0844ebd50079a3c159bf703947fa4caeb3823350a39...,NaN,964,rolling,True,84,4127,38,08:03|15:46|15:56,NaN,0,NaN,v1-2026-10-10
6,A2DqNIw2VPU,https://youtu.be/A2DqNIw2VPU,Le CARDIO a complètement changé mon physique e...,Le CARDIO a complètement changé mon physique e...,ba9abda1eb04bb43a1b81a6a7d8c62493117b3529839f2...,NaN,1047,rolling,True,76,4244,7,13:05|13:22,NaN,0,NaN,v1-2026-10-10
7,-yB8TWvU83g,https://youtu.be/-yB8TWvU83g,5 conseils simples pour gagner ses premiers ki...,5 conseils simples pour gagner ses premiers ki...,4ac28a02662cb44552d9e7bf1fbc9d5f4d627cedce0240...,NaN,1585,rolling,True,208,6077,8,NaN,NaN,0,NaN,v1-2026-10-10
8,7b0irWZcfu0,https://youtu.be/7b0irWZcfu0,7 étapes pour transformer votre physique (et v...,7 étapes pour transformer votre physique (et ...,2874208da74a16b744f6f5a844220f5499a64b213bf7ab...,NaN,879,flat,False,108,3592,0,03:55|04:24|04:32|04:47,NaN,0,NaN,v1-2026-10-10
9,OwPD4u0qOvs,https://youtu.be/OwPD4u0qOvs,Peut-on réellement perdre plus de gras en étan...,Peut-on réellement perdre plus de gras en étan...,6e8f413cf41aba822d2e3ea7bf3dd2895c091e1ee7bbb0...,NaN,614,rolling,False,71,2327,2,NaN,https://www.sciencedirect.com/science/article/...,3,mentioned,v1-2026-10-10


In [56]:
coverage_df

,video_id,title,progression,volume,effort,rest_intervals,fatigue_recovery,deload,plateau,substitution,cycles,frequency_split,break_detraining,pain_safety,beginner,nutrition
0,XFsbtGB1TXQ,Ce format d'entraînement est parfait (Programm...,5,5,3,0,9,0,0,5,2,14,1,0,0,0
1,1OsDiVvQT7k,On s'est tous trompés sur le volume optimal po...,5,39,6,1,7,2,2,1,7,1,1,0,0,0
2,05WPpiDdC78,À quel niveau de musculation êtes-vous vraimen...,2,1,5,0,3,0,1,2,1,3,0,1,4,7
3,_-XnY-bwD4A,7 conseils qui m'ont aidé à avoir un GROS DOS,5,1,0,1,2,0,0,1,0,0,0,0,0,2
4,cvfTTuOZKV8,"Comment j'ai atteint 9,5% de masse grasse en 3...",7,6,0,0,15,0,0,0,2,2,1,0,0,17
5,ZP0LLX_Srs8,Est-ce que je regrette d'avoir perdu autant de...,0,1,1,0,5,0,0,1,0,0,2,0,0,3
6,A2DqNIw2VPU,Le CARDIO a complètement changé mon physique e...,16,3,5,0,3,0,2,0,0,5,0,2,1,2
7,-yB8TWvU83g,5 conseils simples pour gagner ses premiers ki...,4,0,10,0,1,0,1,1,0,1,2,1,0,0
8,7b0irWZcfu0,7 étapes pour transformer votre physique (et v...,7,1,4,1,1,0,1,0,0,0,0,0,1,0
9,OwPD4u0qOvs,Peut-on réellement perdre plus de gras en étan...,1,2,3,0,2,0,0,0,0,2,0,0,1,18


In [ ]:
TOPIC_MIN_COUNT = 1
NON_TOPIC_COLS = {"video_id", "title"}


def final_metadata_match(chunk_dict, coverage_df, manifest_df,
                         topic_min_count=TOPIC_MIN_COUNT,
                         speaker="Nassim Sahili"):
    """Attach video-level metadata to every chunk of one video.
    Returns a list of flat chunk records."""

    video_id = next(iter(chunk_dict.values()))["video_id"]

    cov = coverage_df.loc[coverage_df["video_id"] == video_id]
    man = manifest_df.loc[manifest_df["video_id"] == video_id]

    cov_row, man_row = cov.iloc[0], man.iloc[0]

    topic_cols = [c for c in coverage_df.columns if c not in NON_TOPIC_COLS]
    topic_counts = {t: int(cov_row[t]) for t in topic_cols}
    topics = [t for t, n in topic_counts.items() if n >= topic_min_count]

    refs_raw = man_row["scientific_references"]
    references = [] if pd.isna(refs_raw) or not str(refs_raw).strip() \
        else str(refs_raw).split("|")

    review_status = man_row["reference_review_status"]
    review_status = None if pd.isna(review_status) else review_status


    records = []
    for chunk in chunk_dict.values():
        records.append({
            **chunk,
            "start": int(chunk["start"]),
            "end": int(chunk["end"]),
            "start_url": f'{chunk["video_url"]}?t={int(chunk["start"])}',
            "title": man_row["title"],
            "speaker": speaker,
            "topics": topics,
            "topic_counts": topic_counts,
            "scientific_references": references,
            "reference_review_status": review_status,
        })
    return records